## Embeddings with NNsight

We generate the sentence embeddings in NNsight by wrapping the BERT model and calling the library. We will show that the embeddings produced this way are qualitatively identical to those produced by simply grabbing `hidden_states[-1]` from the model. The use of direct access to model interals for the sentence embeddings needed in experiments is only done due to speed concerns. Using NNsight incurs about a 4x speed penalty on the machine we used to produce embeddings.

In [23]:
import torch
from transformers import AutoTokenizer, AutoModel
from nnsight import LanguageModel

# we should try to use the GPU since this is a heavy task
# use CPU by default, so the program still runs if no GPU is available
dev = 'cpu'
if torch.cuda.is_available():
    # but, if we have a GPU, try to use it
    dev = 'cuda'

In [ ]:
# load bert
bert = LanguageModel('google-bert/bert-base-uncased',  
                     automodel=AutoModel,
                     device_map=dev) # device map here
                                     # can't .to(dev) with NNsight's LanguageModel

# load the tokenizer for bert
bert_tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased')

/home/anonymized/Desktop/blackbox-re26/.venv/lib/python3.12/site-packages/nnsight/intervention/envoy.py:758: UserWarning: Module `model.encoder.layer.0.attention` of type `<class 'transformers.models.bert.modeling_bert.BertAttention'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/home/anonymized/Desktop/blackbox-re26/.venv/lib/python3.12/site-packages/nnsight/intervention/envoy.py:758: UserWarning: Module `model.encoder.layer.0` of type `<class 'transformers.models.bert.modeling_bert.BertLayer'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/home/anonymized/Desktop/blackbox-re26/.venv/lib/python3.12/site-packages/nnsight/intervention/envoy.py:758: UserWarning: Module `model.encoder.layer.1.attention` of type `<class 'transformers.models.bert.modeling_bert.BertA

We get a lot of cell output warnings here. Note this line that appears in all of the warnings: `nnsight access for output will be mounted at .nns_output instead of .output for this module only`, which is a likely reason for why working NNsight with BERT may be difficult, if we are ignoring these messages.

In [25]:
import pandas as pd
import numpy as np

# load the dataset created with save_samples.ipynb
# edit this line to process each of the 4 datasets
df = pd.read_parquet('../datasets/books.parquet')

# collect embeddings here
embed = []

In [26]:
for idx, sent in enumerate(df['txt'].tolist()):
    # .to(dev), hopefully passes the tokenization task to the GPU as well
    tok = bert_tokenizer(sent, truncation=True, max_length=512, return_tensors='pt').to(dev)

    # don't calculate gradients for backpropagation
    # this saves some time when running the model
    with torch.no_grad():
        # NNsight call
        with bert.trace(tok):
            last_hidden = bert.encoder.layer[11].nns_output.save()

    # we are looking for token [CLS] which is at position 0
    embed.append(last_hidden[0,0,:].detach().cpu().float()) 

    # grab 1000 samples only
    # we just want to check for differences
    if idx > 1000:
        break

# matrix with all output vectors put together
matrix = torch.stack(embed).cpu().numpy().astype(np.float32)
# save to disk
np.save('../embeds/full_nns/books_embed.npy', matrix)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2338.68it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


We now compare raw BERT outputs with outputs extracted by NNsight. Even when running the same model on the same sentence, twice, we might produce slightly different embeddings for the `[CLS]` token, and as a matter of fact for all of the tokens, due to floating-point arithmetic precision loss. For example, it is nondeterministic if we execute `a+(b+c)` or `(a+b)+c`, and these two values are, famously, not always equal in floating point.

So, we are interested in there being only a very small error across many sentences to show that we do actually produce equivalent, if admittedly not identical, embeddings.

In [36]:
# load NNsight embeddings
new = np.load('../embeds/full_nns/books_embed.npy')

# load original embeddings extracted directly from bert internals
old = np.load('../embeds/full/books_embed.npy', mmap_mode='r')
# and truncate it, taking only as many sentences as in the NNsight sample
# (do recall that we only embedded a couple of samples in the cell above)
old = old[:len(new)]

# calculate errors
delta = np.abs(new - old)

# show that the error is small enough to be insignificant
print(f'delta max={delta.max():.8f}, mean={delta.mean():.8f}, std={delta.std():.8f}')


delta max=0.00000525, mean=0.00000019, std=0.00000018
